# Day 4 — Feature Engineering & Hyperparameter Tuning

    The selected data set from week 3 is the Train dataset. It will be used to dive deep and explore feature engineering concepts.

In [54]:
import pandas as pd

In [55]:
df = pd.read_csv("train.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


    After observing the DataFrame, the selected columns for feature selection are sibsp and parch as they form the family sizes of passengers, and for feature extraction, passenger titles will be extracted from their names.

### Feature Selection

In [56]:
df['family_size'] = df['SibSp'] + df['Parch'] + 1

Forming a family size column will assist learning wether families of larger sizes experienced higher survival rates or the opposite.

In [57]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Ref', 'Sir', 'Jonkheer', 'Dona'],'Rare')
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme':'Mrs'})


Extracting titles from a high cardinality column that is useless to portray the social classes of those who survived.

In [65]:
df=df.copy()
df = df.drop(columns=["Cabin","Name","Ticket","PassengerId"], errors='ignore')


age_med=df["Age"].median()
df["Age"]= df["Age"].fillna(age_med)

df_clean = pd.get_dummies(df,columns=['Sex','Embarked','Pclass','Title'], drop_first=True)


X = df_clean.drop(["Survived"], axis = 1)
y = df_clean["Survived"]


In [66]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Hyperparameter Tunisng - GridSearchCV

    In this section, a hyperparameter grid for a Random Forest Model will be defined and used to train and cross-validate a stratifies K-Fold model to find the best performing model parameters.

In [67]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
"n_estimators": [100, 200, 400],
"max_depth": [5, 7, 10, None],
"min_samples_split": [2,5,8],
"min_samples_leaf": [1,2,4],
"class_weight":[None, 'balanced']
}


skf =StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                   cv=skf,param_grid=param_grid, scoring='f1', n_jobs=1, verbose=1)


grid.fit(X_train, y_train)
print(grid.best_params_) 
print(grid.best_score_) 
best_model = grid.best_estimator_

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
{'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
0.7685946059842766


The F1 score from week 3 was 0.57. In compsrison to the cross-validated score, the later improved in performance -by 0.2- as a result of hyperparameter tuning & feature engineering .

    The gain is attributed to:
        n_setimators
        max_depth - eliminated overfitting
        class_weight - covered for the minority class which improved recall and F1.

    Finally, training a model with hyperparameter tuning is essential for finding the best settings under which the model performs, but doing it manually is an exhausing task. Thanks to automated tuning through GridSearchCV as through it time is saved and effeciency is achieved.